# Tool Calls Fine-tuning

ref: Unsloth

    @software{unsloth,
    author = {Daniel Han, Michael Han and Unsloth team},
    title = {Unsloth},
    url = {https://github.com/unslothai/unsloth},
    year = {2023}
    }

## Install

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Unsloth


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 6144 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128  ##
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  ##
    lora_dropout = 0.05, # Supports any, but = 0 is optimized  ##
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.17 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


### Data Prep

In [ ]:
from unsloth.chat_templates import get_chat_template
# from llama_api import TOOLS

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# add tools definition if required
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    # texts = [tokenizer.apply_chat_template(convo, tools = TOOLS, tokenize = False, add_generation_prompt = False) for convo in convos]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }


In [ ]:
from datasets import load_dataset
# load a single json
dataset = load_dataset("json", data_files="QA_conv4.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from unsloth.chat_templates import standardize_sharegpt
# dataset = standardize_sharegpt(dataset['train'], aliases_for_system=["system"], aliases_for_user=["user"], aliases_for_assistant=["assistant"])
# my data is in standard format, doesn't need standardize_sharegpt，so map directly
dataset = dataset['train'].map(formatting_prompts_func, batched = True,)
# dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/727 [00:00<?, ? examples/s]

In [ ]:
dataset[5]["conversations"]

[{'content': "You are an economic data assistant. \nWhen provided with FRED economic data, analyze it clearly and concisely.\nFocus on trends, key statistics, and answering the user's specific question.",
  'role': 'system'},
 {'content': "How's the real estate market in the past 2 years?",
  'role': 'user'},
 {'content': 'Could you please clarify which specific data points or indicators you would like to analyze regarding the real estate market? For instance, are you interested in new home sales, housing affordability, mortgage rates, or another specific metric?',
  'role': 'assistant'}]

In [ ]:
dataset[5]["text"]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an economic data assistant. \nWhen provided with FRED economic data, analyze it clearly and concisely.\nFocus on trends, key statistics, and answering the user's specific question.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nHow's the real estate market in the past 2 years?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nCould you please clarify which specific data points or indicators you would like to analyze regarding the real estate market? For instance, are you interested in new home sales, housing affordability, mortgage rates, or another specific metric?<|eot_id|>"

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,  ##
        warmup_steps = 10,  ##
        num_train_epochs = 2, # Set this for 1 full training run.
        # max_steps = 279,  ## 279 = 93 * 3 = 3 epochs
        learning_rate = 2e-4,  ##
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,  ##
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/727 [00:00<?, ? examples/s]

We also use Unsloth's train_on_completions method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/727 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/727 [00:00<?, ? examples/s]

Unsloth: Removed 88 out of 727 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


We verify masking is actually done:

In [ ]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\nYou are an economic data assistant. \nWhen provided with FRED economic data, analyze it clearly and concisely.\nFocus on trends, key statistics, and answering the user\'s specific question.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat\'s the trade balance between US and China in 2023?<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n[Tool calls made]: {"name": "get_fred_data", "arguments": {"series_id": "EXPCH", "start_date": "2023-01-01", "end_date": "2023-12-31"}}\n{"name": "get_fred_data", "arguments": {"series_id": "IMPCH", "start_date": "2023-01-01", "end_date": "2023-12-31"}}<|eot_id|><|start_header_id|>ipython<|end_header_id|>\n\n"[{\\"success\\": true, \\"series_id\\": \\"EXPCH\\", \\"indicator_name\\": \\"Exports to China\\", \\"description\\": \\"Exports to China measures the total value of goods and services that

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

training results

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 639 | Num Epochs = 2 | Total steps = 160
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.821100
2,0.705500
3,0.583900
4,0.495500
5,0.743000
6,0.615300
7,0.722700
8,0.606600
9,0.748600
10,0.751100


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
# from unsloth.chat_templates import get_chat_template

# tokenizer = get_chat_template(
#     tokenizer,
#     chat_template = "llama-3.1",
# )
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#       {
#         "role": "system",
#         "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
#       },
#       {
#         "role": "user",
#         "content": "How has consumer sentiment and saving behavior changed since 2024?"
#       }]

# # add tools
# inputs = tokenizer.apply_chat_template(
#     messages,
#     # tools = TOOLS,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
#                          temperature = 1.5, min_p = 0.1)
# tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# messages = [
#       {
#         "role": "system",
#         "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
#       },
#       {
#         "role": "user",
#         "content": "How has consumer sentiment and saving behavior changed since 2024?"
#       }]
# inputs = tokenizer.apply_chat_template(
#     messages,
#     # tools = TOOLS,
#     tokenize = True,
#     add_generation_prompt = True, # Must add for generation
#     return_tensors = "pt",
# ).to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer, skip_prompt = True)
# _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
#                    use_cache = True, temperature = 1.5, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('llama_lora/tokenizer_config.json',
 'llama_lora/special_tokens_map.json',
 'llama_lora/chat_template.jinja',
 'llama_lora/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
      {
        "role": "system",
        "content": "You are an economic data assistant with access to FRED API. Available FRED Economic Indicators:\n\nFormat: - SERIES_ID - Indicator Name - Unit\n- TREAST - US Treasuries Held by the Fed - Billions of Dollars\n- WSHOMCB - Mortgage Backed Sec Held by the Fed - Millions of Dollars\n- WALCL - All Fed Reserve Banks - Total Assets - Millions of Dollars\n- TOTBKCR - Bank Credit of All Commercial Banks - Billions of Dollars\n- TOTALSEC - Securitized Total Consumer Loans - Millions of Dollars\n- TOTALSL - Total Consumer Credit Outstanding - Millions of Dollars\n- INVEST - Total Investments All Commercial Banks - Millions of Dollars\n- USGSEC - US Treasury and Agency Securities at All Commercial Banks - Billions of Dollars\n- CONSUMER - Total Consumer Loans - Millions of Dollars\n- BUSLOANS - Total Commercial/Industrial Loans - Billions of Dollars\n- DALLCACBEP - Delinquencies On All Loans And Leases - Billions of Dollars\n- T10Y2Y - US 10-YR / 2-YR Spread - Percent\n- TB3MS - 3-Month T-Bill: Secondary Market Rate - Percent\n- DGS10 - 10-Yr Treasury Const. Maturity Rate - Percent\n- GFDEBTN - Federal Government Debt (Public) - Millions of Dollars\n- FYOINT - Interest on National Debt - Millions of Dollars\n- FYONET - Federal Spending - Millions of Dollars\n- FYFR - Federal Receipts - Millions of Dollars\n- FYFSD - Budget Deficit/Surplus - Millions of Dollars\n- CDSP - Consumer Debt/Income Ratio - Percent\n- PERMIT - New Home Permits - Thousands of Units\n- HSN1F - New Home Sales - Thousands\n- CMDEBT - Outstanding Mortgage Debt - Millions of Dollars\n- DGORDER - Manufacturers' New Orders - Millions of Dollars\n- TCU - Capacity Utilization: Total Industry - Percent\n- TTLCONS - Total Construction Spending - Millions of Dollars\n- BUSINV - Total Business Inventories - Millions of Dollars\n- ALTSALES - Light Weight Vehicle Sales - Millions of Units\n- UMCSENT - Univ of Michigan: Consumer Sentiment - Index 1966:Q1=100\n- STLFSI - St. Louis Financial Stress Index - Index\n- WTISPLC - Spot Oil Price - West Texas Intermediate - Dollars per Barrel\n- CPIAUCSL - Consumer Price Index: Seasonally Adj. - Index 1982-1984=100\n- UNRATE - Civilian Total Unemployment Rate - Percent\n- UEMP27OV - Long Term Unemployment: >27 WKS - Thousands of Persons\n- UEMPMED - Length of Unemployment - Weeks\n- CE16OV - Total US Workforce - Thousands of Persons\n- EMRATIO - US Employment/Population Ratio - Percent\n- POP - US Population - Thousands\n- AHEMAN - Avg Hourly Earnings: Manufacturing - Dollars per Hour\n- AWHMAN - Avg Weekly Hours: Manufacturing - Hours\n- AWOTMAN - Avg Weekly OT Hours: Manufacturing - Hours\n- DEXUSUK - USD/GBP Currency Exchange Rate - U.S. Dollars to One U.K. Pound Sterling\n- DEXUSEU - USD/EUR Currency Exchange Rate - U.S. Dollars to One Euro\n- DEXJPUS - JPN/USD Currency Exchange Rate - Japanese Yen to One U.S. Dollar\n- DEXMXUS - MXP/USD Currency Exchange Rate - Mexican Pesos to One U.S. Dollar\n- DEXCAUS - CAD/USD Currency Exchange Rate - Canadian Dollars to One U.S. Dollar\n- DEXCHUS - CNY/USD Currency Exchange Rate - Chinese Yuan Renminbi to One U.S. Dollar\n- COMPOUT - Commercial Paper Outstanding - Billions of Dollars\n- VIXCLS - CBOE Volatility Index - Index\n- GDP - US Gross Domestic Product - Billions of Dollars\n- GNP - US Gross National Product - Billions of Dollars\n- GDI - US Gross Domestic Income - Billions of Dollars\n- NETFI - US Current Account Balance - Billions of Dollars\n- EXPGS - US Exports Goods & Services - Billions of Dollars\n- IMPGS - US Imports Goods & Services - Billions of Dollars\n- DGI - Fed Govt: Defense Budget - Billions of Dollars\n- FGRECPT - Fed Govt: Tax Receipts - Billions of Dollars\n- TGDEF - Fed Govt: Budget Deficit - Billions of Dollars\n- CP - Corporate Profits After Tax - Billions of Dollars\n- DIVIDEND - Corporate Dividends - Billions of Dollars\n- PI - Personal Income - Billions of Dollars\n- PSAVE - Personal Savings - Billions of Dollars\n- PCE - Personal Consumption Expenditures - Billions of Dollars\n- PSAVERT - Personal Savings Rate - Percent\n- MORTGAGE30US - 30-yr Conventional Mortgage Rate - Percent\n- DPCREDIT - Discount Rate - Percent\n- FEDFUNDS - Effective Federal Funds Rate - Percent\n- M1 - M1 Money Supply - Billions of Dollars\n- M2 - M2 Money Supply - Billions of Dollars\n- M1V - Velocity of M1 Money Stock - Ratio\n- M2V - Velocity of M2 Money Stock - Ratio\n- PPIACO - Producer Price Index: All Commodities - Index 1982=100\n- IMPCH - Imports from China - Millions of Dollars\n- IMPJP - Imports from Japan - Millions of Dollars\n- IMPMX - Imports from Mexico - Millions of Dollars\n- IMPCA - Imports from Canada - Millions of Dollars\n- IMPGE - Imports from Germany - Millions of Dollars\n- IMPUK - Imports from UK - Millions of Dollars\n- EXPCH - Exports to China - Millions of Dollars\n- EXPJP - Exports to Japan - Millions of Dollars\n- EXPMX - Exports to Mexico - Millions of Dollars\n- EXPCA - Exports to Canada - Millions of Dollars\n- EXPGE - Exports to Germany - Millions of Dollars\n- EXPUK - Exports to UK - Millions of Dollars\n- BOPGEXP - Exports: Goods - Millions of Dollars\n- BOPGIMP - Imports: Goods - Millions of Dollars\n- BOPGTB - Balance: Goods - Millions of Dollars\n- BOPSIMP - Imports: Services - Millions of Dollars\n- BOPSTB - Balance: Services - Millions of Dollars\n- BOPGSTB - Balance: Goods & Services - Millions of Dollars\nNote: (M)=Monthly, (Q)=Quarterly, (W)=Weekly, (D)=Daily, (Y)=Yearly\n\nWhen asked about economic data, use the get_fred_data function with the appropriate series_id.\n\nYou will receive data from ALL tool calls.\n\nIMPORTANT: \n1. Always specify start_date and end_date based on the user's question. Always use YYYY-MM-DD format, NOT relative dates like \"-2y\"\n2. If no time period specified, use recent 1 year by default\n3. Each series_id should only be called ONCE with a single continuous date range.\n   - WRONG: calling GDP twice with 2018-2020 and 2021-2023\n   - RIGHT: calling GDP once with 2018-2023\n4. Today is 2026-02-25\n"
      },
      {
        "role": "user",
        "content": "How has consumer sentiment and saving behavior changed since 2024?"
      }]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


I can provide the most recent consumer sentiment and savings data for 2024 and compare them to more recent figures to assess any changes. 

To begin, I will retrieve the data for “University of Michigan: Consumer Sentiment (Index 1966:Q1=100) (Series ID: UMCSENT)” from the end of 2024.

After retrieval of all necessary indicators, please note that I'll display the requested information from those retrieved series for both Consumer Sentiment and Personal Savings (Series ID: PSAVERT) on the consumer behavior side.

The total U.S. population in millions has also been retrieved to understand


You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if True: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [04:32<00:00, 136.05s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [06:04<00:00, 182.22s/it]


Unsloth: Merge process complete. Saved to `/content/llama_finetune`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama_finetune_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['llama_finetune_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model llama_finetune_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to llama_finetune_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f llama_finetune_gguf/Modelfile


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import shutil
import os

source_path = "llama_finetune_gguf"
destination_path = "/content/drive/MyDrive/llama_finetune_gguf"

# Create the destination directory if it doesn't exist
if not os.path.exists(os.path.dirname(destination_path)):
    os.makedirs(os.path.dirname(destination_path))

# Copy the directory
shutil.copytree(source_path, destination_path, dirs_exist_ok=True)

print(f"'{source_path}' copied to '{destination_path}' successfully.")


'llama_finetune_gguf' copied to '/content/drive/MyDrive/llama_finetune_gguf' successfully.
